# Amazon Nova Act SDK를 활용한 기본 Browser tool 사용법

## 개요

이 튜토리얼에서는 Nova Act SDK와 Amazon Bedrock AgentCore Browser tool을 함께 사용하는 방법을 알아봅니다. Browser tool을 headless 방식으로 사용하는 예제와 브라우저 화면을 실시간으로 확인하는 예제를 살펴봅니다.


### 튜토리얼 세부 정보


| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                           |
| Agent 유형          | 단일                                                                             |
| Agentic Framework   | Nova Act                                                                         |
| LLM 모델            | Amazon Nova Act model                                                            |
| 튜토리얼 구성 요소  | NovaAct를 사용해 headless 방식으로 Browser tool과 상호 작용                      |
| 튜토리얼 분야       | 범용                                                                             |
| 예제 난이도         | 쉬움                                                                             |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK, Nova Act                                     |

### 튜토리얼 아키텍처

이 튜토리얼에서는 Nova Act와 Browser tool을 함께 사용하는 방법을 설명합니다.  

예제에서는 Nova Act Agent에 자연어 지시를 보내 Bedrock AgentCore Browser에서 headless 방식으로 작업을 수행합니다.

### 튜토리얼 주요 기능

* Browser tool을 headless 방식으로 사용
* Nova Act와 Browser tool을 함께 사용

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.
* Python 3.10+
* AWS 자격 증명. IAM 역할/사용자에 다음 권한이 있어야 합니다. https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html#browser-credentials-config
* Amazon Bedrock AgentCore SDK
* Nova Act SDK 및 API 키

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## NovaAct와 Bedrock AgentCore Browser tool 함께 사용하기
로컬에서 실행할 수 있는 Python 스크립트를 생성합니다. 스크립트에서 Nova Act는 브라우저 세션의 CDP endpoint를 사용해 연결한 후 Playwright 액션을 수행합니다.

In [ ]:
%%writefile basic_browser_with_nova_act.py
"""Amazon Bedrock AgentCore와 Nova Act를 사용하는 브라우저 자동화 스크립트입니다.

이 스크립트는 다음과 같은 AI 기반 웹 자동화를 보여 줍니다:
- Amazon Bedrock AgentCore를 통한 브라우저 세션 초기화
- 자연어 웹 상호 작용을 위한 Nova Act 연결
- 브라우저를 사용한 자동 검색 및 데이터 추출
"""

from bedrock_agentcore.tools.browser_client import browser_session
from nova_act import NovaAct
from rich.console import Console
import argparse
import json

console = Console()

from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print("using region", region)

def browser_with_nova_act(prompt, starting_page, nova_act_key, region="us-west-2"):
    result = None
    with browser_session(region) as client:
        ws_url, headers = client.generate_ws_headers()
        try:
            with NovaAct(
                cdp_endpoint_url=ws_url,
                cdp_headers=headers,
                preview={"playwright_actuation": True},
                nova_act_api_key=nova_act_key,
                starting_page=starting_page,
            ) as nova_act:
                result = nova_act.act(prompt)
        except Exception as e:
            console.print(f"NovaAct error: {e}")
        finally:
            return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True, help="Browser Search instruction")
    parser.add_argument("--starting-page", required=True, help="Starting URL")
    parser.add_argument("--nova-act-key", required=True, help="Nova Act API key")
    parser.add_argument("--region", default="us-west-2", help="AWS region")
    args = parser.parse_args()

    result = browser_with_nova_act(
        args.prompt, args.starting_page, args.nova_act_key, args.region
    )
    console.print(f"\n[cyan] Response[/cyan] {result.response}")
    console.print(f"\n[bold green]Nova Act Result:[/bold green] {result}")

#### 스크립트 실행하기
스크립트를 실행하기 전에 아래에 Nova Act API 키를 입력합니다.

In [ ]:
NOVA_ACT_KEY = ""  ### 여기에 Nova Act API 키를 입력하세요

In [ ]:
!python basic_browser_with_nova_act.py --prompt "Search for macbooks and extract the details of the first one" --starting-page "https://www.amazon.com/" --nova-act-key {NOVA_ACT_KEY}

In [ ]:
!python basic_browser_with_nova_act.py --prompt "Extract and return Amazon revenue for the last 4 years" --starting-page "https://stockanalysis.com/stocks/amzn/financials/" --nova-act-key {NOVA_ACT_KEY}

## 내부에서는 어떤 일이 일어났을까요?

* `browser_session`으로 브라우저 클라이언트를 인스턴스화하면 Browser client가 생성되고 세션이 시작됩니다.
* 그런 다음 `cdp_endpoint_url`과 `cdp_headers`를 사용해 Nova Act가 해당 브라우저 세션을 가리키도록 구성했습니다.
* Nova Act SDK는 자연어 지시를 받아 브라우저에서 Playwright 액션을 생성했습니다.

# 축하합니다!